# SBA loan risk — data cleaning and preparation

This is my part of the SBA case study. I used the original `SBAnational.csv` to clean up the data and get it ready for our team's charts, financial analysis, and logistic regression model.

The main thing we're trying to predict is whether a loan was **paid in full** (`PIF`) or **charged off** (`CHGOFF`). For the model, a charge-off counts as `1` and a paid-in-full loan counts as `0`. If there's no status recorded, we won't guess.

I'm keeping the actual model training in a separate notebook. My job here is to make sure everyone starts with consistent data, and to avoid including information that a bank would only learn *after* approving the loan.


## What this notebook does

It starts with the original CSV, checks for missing or strange values, cleans the formats, and saves two files:

- **`sba_eda_ready.csv`** is for exploring the data. It keeps all the records and includes historical loan outcomes and disbursement amounts for charts and the class financial exercise.
- **`sba_model_candidates.csv`** is for predicting default at the approval stage. It keeps only loans with known outcomes and removes the obvious post-approval fields from the possible predictors.

The notebook also creates audit files in `outputs/cleaning_audit/` so the team can see what was changed and how many rows were affected.

**To run it:** Put `SBAnational.csv` in `data/raw/`, open the notebook in VS Code or Jupyter, and choose **Run All**. If your CSV is somewhere else, set `RAW_CSV_OVERRIDE` in the next code cell. Keep the original file untouched.


### What I got when I tested the supplied CSV

These are the results from the original file used for this project. If we rerun the notebook with a different version of the data, the audit reports will update too.

| Check | Result |
| --- | ---: |
| Original records | **899,164** |
| Paid in full | **739,609** |
| Charged off | **157,558** |
| Status missing (not used for modeling) | **1,997** |
| Rows available for modeling | **897,167** |
| Duplicate loan IDs | **0** |
| NAICS listed as unknown/0 | **201,948** |
| Loan term is zero or negative | **810** |
| Unusual fiscal-year labels | **18** |
| Dates corrected for a two-digit-year issue | **9** |
| Approval date/fiscal-year mismatches after checking | **54** |

The counts are in the audit files too; this table is just an overview.


In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
from IPython.display import display

# Change only this if you keep the CSV outside data/raw/.
RAW_CSV_OVERRIDE = None  # Example: r"C:\Users\me\Downloads\SBAnational.csv"

cwd = Path.cwd().resolve()
search_roots = [cwd, *list(cwd.parents)[:3]]
repo_root = next(
    (p for p in search_roots if (p / 'notebooks').is_dir() and (p / 'data').is_dir()),
    cwd
)

paths_to_try = (
    [Path(RAW_CSV_OVERRIDE).expanduser()] if RAW_CSV_OVERRIDE
    else [repo_root / 'data' / 'raw' / 'SBAnational.csv',
          repo_root / 'SBAnational.csv', cwd / 'SBAnational.csv']
)
raw_path = next((p for p in paths_to_try if p.is_file()), None)
if raw_path is None:
    raise FileNotFoundError(
        'SBAnational.csv not found. Place it in data/raw/ or set RAW_CSV_OVERRIDE.\n'
        + '\n'.join(str(p) for p in paths_to_try)
    )

processed_dir = repo_root / 'data' / 'processed'
audit_dir = repo_root / 'outputs' / 'cleaning_audit'
processed_dir.mkdir(parents=True, exist_ok=True)
audit_dir.mkdir(parents=True, exist_ok=True)
print('Input:', raw_path)
print('Project root:', repo_root)

## 1. Load the original data

First, I check that the CSV still has the expected 27 columns. I keep things like ZIP codes and loan IDs as text so leading zeros aren't lost. Nothing in this step changes the original CSV.


In [ ]:
CODE_COLUMNS = {
    'LoanNr_ChkDgt': 'string',
    'Zip': 'string',
    'NAICS': 'string',
    'ApprovalFY': 'string',
    'NewExist': 'string',
    'UrbanRural': 'string',
    'FranchiseCode': 'string',
    'LowDoc': 'string',
    'RevLineCr': 'string',
}
try:
    raw = pd.read_csv(raw_path, dtype=CODE_COLUMNS, encoding='utf-8', low_memory=False)
except UnicodeDecodeError:
    raw = pd.read_csv(raw_path, dtype=CODE_COLUMNS, encoding='latin-1', low_memory=False)

expected = [
    'LoanNr_ChkDgt','Name','City','State','Zip','Bank','BankState',
    'NAICS','ApprovalDate','ApprovalFY','Term','NoEmp','NewExist',
    'CreateJob','RetainedJob','FranchiseCode','UrbanRural','RevLineCr',
    'LowDoc','ChgOffDate','DisbursementDate','DisbursementGross',
    'BalanceGross','MIS_Status','ChgOffPrinGr','GrAppv','SBA_Appv'
]
missing_cols = sorted(set(expected) - set(raw.columns))
if missing_cols:
    raise ValueError('Unexpected source schema: missing ' + str(missing_cols))
if len(raw.columns) != 27:
    print('WARNING: Expected 27 columns; found', len(raw.columns))

n_original = len(raw)
print(f'Loaded {n_original:,} records, {raw.shape[1]} columns.')
print('Duplicate loan IDs:', int(raw['LoanNr_ChkDgt'].duplicated(keep=False).sum()))
display(raw.head(3)[['LoanNr_ChkDgt','ApprovalDate','ApprovalFY','Term','MIS_Status','GrAppv']])

## 2. See what's missing or inconsistent

Before changing anything, I check missing values and look at the codes used in columns like `LowDoc` and `RevLineCr`. This makes it easier to explain why a value was cleaned or left alone later. I don't want to delete a loan just because one column is blank.


In [ ]:
# Treat empty/whitespace-only strings as missing for auditing.
def blank_count(s):
    if pd.api.types.is_string_dtype(s) or pd.api.types.is_object_dtype(s):
        return int(s.isna().sum() + s.fillna('').astype(str).str.strip().eq('').sum() - s.isna().sum())
    return int(s.isna().sum())

raw_audit = pd.DataFrame({
    'column': raw.columns,
    'dtype_original': [str(raw[c].dtype) for c in raw.columns],
    'missing_n': [blank_count(raw[c]) for c in raw.columns],
    'unique_nonnull': [raw[c].nunique(dropna=True) for c in raw.columns]
})
raw_audit['missing_pct'] = (100 * raw_audit.missing_n / n_original).round(3)
raw_audit.sort_values('missing_pct', ascending=False).to_csv(
    audit_dir / '01_original_missingness.csv', index=False
)

for c in ['MIS_Status', 'LowDoc', 'RevLineCr', 'NewExist', 'UrbanRural']:
    profile = raw[c].fillna('<MISSING>').astype(str).str.strip().value_counts(dropna=False)
    profile.rename_axis('original_value').reset_index(name='loans').to_csv(
        audit_dir / ('02_codes_' + c + '.csv'), index=False
    )
print('Most-missing source fields:')
display(raw_audit.sort_values('missing_pct', ascending=False).head(10))
print('Original statuses:')
display(raw['MIS_Status'].fillna('<MISSING>').value_counts(dropna=False).to_frame('loans'))

## 3. Clean up the columns

Here I make dates, dollar amounts, and category labels more consistent. A few decisions matter:

- `NAICS = 0` means the industry isn't specified, not that the loan belongs to industry zero.
- If a category has an unfamiliar code, I mark the cleaned version as unknown instead of assuming what the code means.
- `ApprovalFY` uses the federal fiscal year, so an approval in October–December can belong to the *next* fiscal year. The code also checks dates that were accidentally read as being in the 2060s.
- I keep missing loan outcomes as missing. They are not automatically counted as paid-in-full loans.

I'm also careful about timing: an approval-date feature may be useful, but a later charge-off amount or disbursement date would reveal information the bank didn't have when approving the loan.


In [ ]:
# Two high-cardinality name fields are not useful for this handoff and may expose
# borrower/lender identities. Keep them in the untouched raw input only.
eda = raw.drop(columns=['Name', 'Bank']).copy()

# Stable record identifier: keep as text; DO NOT use as a predictor.
eda['LoanNr_ChkDgt'] = eda['LoanNr_ChkDgt'].astype('string').str.strip()

# Normalize statuses, but never assume that missing/unknown means 'repaid'.
status_raw = eda['MIS_Status'].astype('string').str.upper().str.replace(r'\s+', '', regex=True)
status_normalized = status_raw.map({'PIF': 'PIF', 'CHGOFF': 'CHGOFF'})
eda['MIS_Status'] = status_normalized.astype('string')
eda['Default'] = pd.Series(pd.NA, index=eda.index, dtype='Int8')
eda.loc[eda['MIS_Status'].eq('PIF').fillna(False), 'Default'] = 0
eda.loc[eda['MIS_Status'].eq('CHGOFF').fillna(False), 'Default'] = 1

# Dollar fields. Charge-off amount, balance, and actual disbursement are for
# DESCRIPTIVE/FINANCIAL analysis, never predictive approval-time inputs.
money_fields = ['DisbursementGross', 'BalanceGross', 'ChgOffPrinGr', 'GrAppv', 'SBA_Appv']
for col in money_fields:
    s = (eda[col].astype('string')
         .str.replace(r'[$,\s]', '', regex=True)
         .replace('', pd.NA))
    eda[col] = pd.to_numeric(s, errors='coerce').astype('float64')

# Dates. Two-digit years use the input's DD-Mon-YY convention.
date_cols = ['ApprovalDate', 'DisbursementDate', 'ChgOffDate']
date_century_corrections = {}
for col in date_cols:
    parsed = pd.to_datetime(eda[col], format='%d-%b-%y', errors='coerce')
    # Python maps years 00-68 to 2000-2068. In this historical dataset,
    # 2061-2068 records are 1961-1968, not future loans.
    wrong_century = parsed.dt.year.gt(2026)
    date_century_corrections[col] = int(wrong_century.sum())
    if wrong_century.any():
        parsed.loc[wrong_century] = parsed.loc[wrong_century] - pd.DateOffset(years=100)
    eda[col] = parsed

# Extract 4-digit fiscal year from the original string; don't hide malformed labels.
fy_raw = raw['ApprovalFY'].astype('string').str.strip()
eda['ApprovalFY'] = pd.to_numeric(
    fy_raw.str.extract(r'(\d{4})', expand=False), errors='coerce'
).astype('Int64')
eda['ApprovalYear'] = eda['ApprovalDate'].dt.year.astype('Int64')
eda['ApprovalYear'] = eda['ApprovalYear'].fillna(eda['ApprovalFY'])
eda['ApprovalMonth'] = eda['ApprovalDate'].dt.month.astype('Int64')

# Preserve 5-digit ZIP strings, padding only digits that were truncated.
zip_raw = eda['Zip'].astype('string').str.strip()
eda['ZIP5'] = zip_raw.where(zip_raw.str.fullmatch(r'\d{1,5}').fillna(False))
eda['ZIP5'] = eda['ZIP5'].str.zfill(5)

# States: unknown if missing/malformed. Unknown != in-state.
for col in ['State', 'BankState']:
    s = eda[col].astype('string').str.upper().str.strip()
    eda[col] = s.where(s.str.fullmatch(r'[A-Z]{2}').fillna(False), 'Unknown')
known_states = eda['State'].ne('Unknown') & eda['BankState'].ne('Unknown')
eda['InState_Bank'] = pd.Series(pd.NA, index=eda.index, dtype='Int8')
eda.loc[known_states, 'InState_Bank'] = (
    eda.loc[known_states, 'State'].eq(eda.loc[known_states, 'BankState'])
).astype('int8')

# NAICS industry sector: code 0 means unknown; preserve as categorical text.
naics_raw = eda['NAICS'].astype('string').str.strip()
naics_valid = naics_raw.str.fullmatch(r'\d{2,6}').fillna(False) & ~naics_raw.str.fullmatch(r'0+').fillna(False)
eda['NAICS_Sector'] = naics_raw.str[:2].where(naics_valid, 'Unknown')

newex = eda['NewExist'].astype('string').str.strip()
eda['NewExist_clean'] = newex.map({'1': 'Existing', '2': 'New'}).fillna('Unknown')
urbr = eda['UrbanRural'].astype('string').str.strip()
eda['UrbanRural_clean'] = urbr.map({'1': 'Urban', '2': 'Rural'}).fillna('Unknown')

for source, target in [('LowDoc', 'LowDoc_clean'), ('RevLineCr', 'RevLineCr_clean')]:
    z = eda[source].astype('string').str.upper().str.strip()
    eda[target] = z.where(z.isin(['Y', 'N']).fillna(False), 'Unknown')

# Numeric counts: preserve abnormal or missing values in EDA; flag them later.
for col in ['Term', 'NoEmp', 'CreateJob', 'RetainedJob']:
    eda[col] = pd.to_numeric(eda[col], errors='coerce')

# A potentially useful ratio if proposed/approved terms are known at scoring time.
eda['SBA_Guarantee_Share'] = (
    eda['SBA_Appv'] / eda['GrAppv'].where(eda['GrAppv'].gt(0))
)
eda.loc[~eda['SBA_Guarantee_Share'].between(0, 1), 'SBA_Guarantee_Share'] = np.nan

print('Cleaned example (no row exclusions yet):')
display(eda[['LoanNr_ChkDgt','MIS_Status','Default','ApprovalYear',
             'NAICS_Sector','GrAppv','SBA_Appv','SBA_Guarantee_Share']].head(5))

## 4. Check questionable records

Now I count anything that needs another look, like unusually coded years, zero terms, or large values. These checks don't automatically delete the loans. Some outliers are real; the point is to know they're there and let the team make a justified decision.


In [ ]:
# These are flags for analysis, not blanket instructions to discard data.
raw_status_nonempty = status_raw.notna() & status_raw.ne('')
unknown_status = raw_status_nonempty & eda['MIS_Status'].isna()
fy_malformed = fy_raw.notna() & fy_raw.ne('') & ~fy_raw.str.fullmatch(r'\d{4}').fillna(False)
# ApprovalFY is a FEDERAL FISCAL YEAR, not the approval calendar year.
# A date in Oct-Dec usually belongs to the following fiscal year.
expected_fiscal_year = (
    eda['ApprovalDate'].dt.year.astype('Int64')
    + eda['ApprovalDate'].dt.month.ge(10).astype('Int64')
)
year_mismatch = (
    eda['ApprovalDate'].notna() & eda['ApprovalFY'].notna()
    & expected_fiscal_year.ne(eda['ApprovalFY'])
)

audit_checks = {
    'raw_records': n_original,
    'unique_loan_ids': int(eda['LoanNr_ChkDgt'].nunique(dropna=True)),
    'duplicate_loan_id_rows': int(eda['LoanNr_ChkDgt'].duplicated(keep=False).sum()),
    'paid_in_full_PIF': int(eda['Default'].eq(0).sum()),
    'charged_off_CHGOFF': int(eda['Default'].eq(1).sum()),
    'excluded_from_supervised_model_missing_or_unknown_target': int(eda['Default'].isna().sum()),
    'unknown_nonempty_status_codes': int(unknown_status.sum()),
    'NAICS_zero_or_unknown': int(eda['NAICS_Sector'].eq('Unknown').sum()),
    'Term_zero_or_negative': int(eda['Term'].le(0).sum()),
    'NoEmp_negative': int(eda['NoEmp'].lt(0).sum()),
    'ApprovalFY_malformed_text': int(fy_malformed.sum()),
    'ApprovalDate_parse_fail': int(eda['ApprovalDate'].isna().sum()),
    'ApprovalFY_vs_expected_federal_fiscal_year_mismatch': int(year_mismatch.sum()),
    'ApprovalDate_1960s_century_rollover_corrected': int(date_century_corrections['ApprovalDate']),
    'Approved_amount_missing_or_nonpositive': int(eda['GrAppv'].isna().sum() + eda['GrAppv'].le(0).sum()),
    'SBA_approval_above_total_approval': int((eda['SBA_Appv'] > eda['GrAppv']).sum()),
    'Actual_disbursement_missing_or_nonpositive': int(eda['DisbursementGross'].isna().sum() + eda['DisbursementGross'].le(0).sum()),
    'ApprovalYear_before_1987': int(eda['ApprovalYear'].lt(1987).sum()),
    'ApprovalYear_after_2014': int(eda['ApprovalYear'].gt(2014).sum()),
    'Unknown_RevLineCr': int(eda['RevLineCr_clean'].eq('Unknown').sum()),
    'Unknown_LowDoc': int(eda['LowDoc_clean'].eq('Unknown').sum()),
}
quality = pd.DataFrame(list(audit_checks.items()), columns=['quality_check', 'rows_or_count'])
quality.to_csv(audit_dir / '03_quality_checks.csv', index=False)
print('Quality checks (flags, not automatic deletions):')
display(quality)

status_counts = eda['MIS_Status'].fillna('Missing/unknown').value_counts(dropna=False)
status_counts.rename_axis('status').reset_index(name='loans').to_csv(
    audit_dir / '04_status_counts.csv', index=False
)

## 5. Save separate files for EDA and modeling

The EDA file can include loan outcomes and disbursement amounts because those help us describe what happened historically. **It should not be used as-is for a loan-approval prediction model.**

For modeling, I keep a smaller set of possible inputs and remove later information such as `ChgOffDate`, `ChgOffPrinGr`, `DisbursementDate`, and `DisbursementGross`. Loan IDs and names are not predictors either.

One detail to confirm with the modeling teammate: `GrAppv`, `SBA_Appv`, and the guarantee share only make sense if the approved loan terms are known when the prediction is made. If we're predicting risk earlier, they should also be removed. `ApprovalYear` is included for checking time periods, not as a baseline predictor.


In [ ]:
MODEL_FEATURES = [
    'State','BankState','Term','NoEmp','NewExist_clean','UrbanRural_clean',
    'NAICS_Sector','RevLineCr_clean','LowDoc_clean','InState_Bank',
    'GrAppv','SBA_Appv','SBA_Guarantee_Share'
]
# Export ID and year for joins/holdout logic, not as baseline features.
MODEL_METADATA = ['LoanNr_ChkDgt', 'ApprovalYear']
MODEL_TARGET = 'Default'

# Exclude missing outcomes only from supervised model candidates.
model = eda.loc[eda['Default'].notna(),
                MODEL_METADATA + MODEL_FEATURES + [MODEL_TARGET]].copy()
model[MODEL_TARGET] = model[MODEL_TARGET].astype('int8')

LEAKAGE_BLOCKLIST = {
    'MIS_Status','ChgOffDate','ChgOffPrinGr','BalanceGross',
    'DisbursementDate','DisbursementGross','Name','Bank','CreateJob',
    'RetainedJob','ApprovalDate','City','ZIP5'
}
assert set(MODEL_FEATURES).isdisjoint(LEAKAGE_BLOCKLIST), 'Leaky predictor included.'
assert model[MODEL_TARGET].isin([0, 1]).all()
assert len(eda) == n_original
assert len(model) + int(eda['Default'].isna().sum()) == n_original
if eda['LoanNr_ChkDgt'].isna().any() or eda['LoanNr_ChkDgt'].duplicated().any():
    raise AssertionError('Missing/duplicate loan IDs. Investigate before handoff.')
if eda['MIS_Status'].notna().sum() != len(model):
    raise AssertionError('Status/model row reconciliation failed.')

eda_path = processed_dir / 'sba_eda_ready.csv'
model_path = processed_dir / 'sba_model_candidates.csv'
eda.to_csv(eda_path, index=False, date_format='%Y-%m-%d')
model.to_csv(model_path, index=False)

reconciliation = pd.DataFrame([
    ('Raw rows', n_original),
    ('EDA-ready rows (all original rows)', len(eda)),
    ('Excluded only from supervised model (missing/unknown status)', int(eda.Default.isna().sum())),
    ('Model-candidate rows (known outcome)', len(model)),
    ('Raw columns', raw.shape[1]),
    ('Baseline model feature columns', len(MODEL_FEATURES)),
], columns=['item','count'])
reconciliation.to_csv(audit_dir / '05_reconciliation.csv', index=False)
print('SAVED:', eda_path)
print('SAVED:', model_path)
display(reconciliation)

## 6. Notes for the modeling and EDA teammates

**If you're building the logistic regression model:** Start with `data/processed/sba_model_candidates.csv`. The target is `Default`, where `1` means charged off and `0` means paid in full. The exact input columns are in `outputs/cleaning_audit/06_model_feature_contract.json`.

Keep `LoanNr_ChkDgt` and `ApprovalYear` out of the baseline predictors. Split the data into training and testing sets **before** learning how to fill missing values, encode categories, or scale numbers. Otherwise information from the test set can leak into the training process.

The model results should be checked on held-out loans with a confusion matrix, precision, recall, and F1 or ROC-AUC. Keep in mind this dataset only covers loans that were approved historically, not the ones that were rejected.

**If you're doing EDA:** Use `sba_eda_ready.csv`. For default rates, leave out unknown loan outcomes and show group sizes alongside percentages. The disbursement fields are useful for the separate class financial exercise, but not for approval-time prediction.


In [ ]:
contract = {
    'dataset': 'sba_model_candidates.csv',
    'target': MODEL_TARGET,
    'positive_label': 1,
    'positive_meaning': 'CHGOFF (loan charged off/default)',
    'negative_label': 0,
    'negative_meaning': 'PIF (loan paid in full)',
    'features': MODEL_FEATURES,
    'metadata_exclude_as_baseline_predictors': MODEL_METADATA,
    'approved_terms_caveat': 'GrAppv, SBA_Appv, and SBA_Guarantee_Share assume approved terms are known at the scoring point. Remove them for earlier pre-underwriting decisions.',
    'fit_imputation_encoding_scaling_after_train_test_split': True,
    'excluded_post_approval_or_outcome_fields': sorted(LEAKAGE_BLOCKLIST),
    'status_missing_treatment': 'Excluded from supervised target model only; retained in EDA export.',
    'excluded_not_by_default': 'Outliers, unknown NAICS, invalid codes, and missing values are not blindly discarded.',
    'selection_bias': 'Only approved loans are represented; outcomes of applications never approved are not observed.'
}
(audit_dir / '06_model_feature_contract.json').write_text(json.dumps(contract, indent=2))

dictionary = pd.DataFrame([
    ('LoanNr_ChkDgt', 'Stable loan identifier', 'Join only; NEVER train'),
    ('MIS_Status', 'Historical repayment outcome', 'EDA label; NEVER train'),
    ('Default', 'Binary outcome: 1=CHGOFF, 0=PIF', 'Train TARGET only'),
    ('ApprovalYear', 'Approval date year / fiscal-year fallback', 'Split/cohort check; exclude baseline predictor'),
    ('Term', 'Approved loan duration (months)', 'Candidate predictor'),
    ('NoEmp', 'Reported number of employees', 'Candidate predictor'),
    ('State', 'Borrower state', 'Candidate predictor'),
    ('BankState', 'Lender state', 'Candidate predictor'),
    ('InState_Bank', 'Borrower/lender states match, when both known', 'Candidate predictor'),
    ('NAICS_Sector', 'First two NAICS digits or Unknown', 'Candidate predictor'),
    ('NewExist_clean', 'Existing/New/Unknown business category', 'Candidate predictor'),
    ('UrbanRural_clean', 'Urban/Rural/Unknown', 'Candidate predictor'),
    ('LowDoc_clean', 'Y/N/Unknown; nonstandard codes not guessed', 'Candidate predictor'),
    ('RevLineCr_clean', 'Y/N/Unknown; nonstandard codes not guessed', 'Candidate predictor'),
    ('GrAppv', 'Gross approved loan amount', 'Approved-terms candidate; document scoring point'),
    ('SBA_Appv', 'SBA-guaranteed approved amount', 'Approved-terms candidate; document scoring point'),
    ('SBA_Guarantee_Share', 'SBA_Appv / GrAppv when in [0,1]', 'Approved-terms candidate; document scoring point'),
    ('DisbursementGross', 'Actual amount disbursed', 'Financial/EDA only; POST-approval'),
    ('ChgOffPrinGr', 'Charged-off principal', 'EDA only; reveals outcome'),
    ('ChgOffDate', 'Date of charge-off', 'EDA only; reveals outcome'),
    ('BalanceGross', 'Recorded loan balance', 'EDA only; outcome-related'),
    ('DisbursementDate', 'Actual disbursement date', 'EDA only; post-approval'),
    ('CreateJob', 'Reported jobs to create', 'EDA only until timing verified'),
    ('RetainedJob', 'Reported retained jobs', 'EDA only until timing verified'),
], columns=['variable', 'interpretation', 'allowed_use'])
dictionary.to_csv(audit_dir / '07_data_dictionary.csv', index=False)

# Lightweight machine-readable QA report to be used by the rest of the team.
summary = {
    'source_name': raw_path.name,
    'source_rows': int(n_original),
    'eda_rows': int(len(eda)),
    'model_labeled_rows': int(len(model)),
    'target_1_defaults': int(model['Default'].sum()),
    'target_0_paid': int((model['Default'] == 0).sum()),
    'target_missing_excluded_from_model': int(eda['Default'].isna().sum()),
    'outputs': {'eda': str(eda_path.relative_to(repo_root)), 'model_candidates': str(model_path.relative_to(repo_root))},
    'post_approval_leakage_prevented': True
}
(audit_dir / '08_handoff_summary.json').write_text(json.dumps(summary, indent=2))
print('Handoff documentation saved to:', audit_dir)
display(pd.DataFrame([summary]).drop(columns=['outputs']))

## 7. Before I hand this over

- [ ] Run the full notebook without errors.
- [ ] Check that the raw and EDA files have the same number of rows and that the outcome counts add up.
- [ ] Read `03_quality_checks.csv` and the unusual-code audit files to see if anything needs another look.
- [ ] Share `sba_eda_ready.csv` with the EDA and finance teammates.
- [ ] Share `sba_model_candidates.csv` and `06_model_feature_contract.json` with the modeling teammate.
- [ ] Make sure the finance teammate knows the assignment uses a simplified +5% for repaid loans and −25% for charged-off loans (based on disbursed amount), not a real bank accounting formula.
- [ ] Push the notebook, README, and small audit files to GitHub. Keep the big raw and processed CSVs out of regular Git.

**Short explanation for our presentation:** We kept the original loan records for analysis, only removed unknown outcomes from the modeling data, cleaned up dates and money fields, and made sure the prediction model won't use information learned after the loan was approved.
